<a href="https://colab.research.google.com/github/yersultanp/Adversarial-Diffusion-Distillation/blob/main/diffusion_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diffusion Distillation


## First let's download the teacher model which will be the SD-v1.5

In [ ]:
!pip install diffusers --upgrade
!pip install invisible_watermark transformers accelerate safetensors
!pip install datasets
!pip install einops
!pip install denoising_diffusion_pytorch

from denoising_diffusion_pytorch import Unet
from inspect import isfunction
from einops import rearrange
from functools import partial
from torch import nn, einsum
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data.dataloader as dataloader
from torch.utils.data import TensorDataset
from torch.autograd import Variable
from torchvision import transforms, datasets
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
import random
import math

In [ ]:
# Download the Dataset
!python /datasets/create_teacher_dataset.py --output_dir "./data/" --prompts_file "./prompts.json"

## Initial Code Skeleton

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline, DDIMScheduler
from diffusers.utils.torch_utils import randn_tensor
import copy

# ==========================================
# 1. The Lightweight Student Scheduler (MLP)
# ==========================================
class LearnedStepScheduler(nn.Module):
    """
    A tiny MLP that predicts which timestep 't' to use for the current step.
    Inputs:
      - step_index: Current step number (0 to K-1)
      - latent_stats: Mean and Var of the current latent (simple content descriptor)
    Output:
      - t: A continuous value between 0 and 1000
    """
    def __init__(self, num_steps, input_dim=2):
        super().__init__()
        self.num_steps = num_steps

        # Simple embedding for the step index (0, 1, 2...)
        self.step_embed = nn.Embedding(num_steps, 32)

        # MLP to predict the timestep
        # Input: Step embedding (32) + Latent Stats (2: mean, std)
        self.net = nn.Sequential(
            nn.Linear(32 + input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1), # Output scalar t
            nn.Sigmoid()      # Bound output to [0, 1] then scale to [0, 1000]
        )

    def forward(self, step_idx, latents):
        # 1. Extract simple stats from latents to make the schedule adaptive
        # Shape: [Batch, Channels, H, W] -> [Batch, 2]
        mean = latents.mean(dim=[1, 2, 3], keepdim=True).squeeze(-1).squeeze(-1)
        std = latents.std(dim=[1, 2, 3], keepdim=True).squeeze(-1).squeeze(-1)
        stats = torch.cat([mean, std], dim=1)

        # 2. Get step embedding
        step_idx_tensor = torch.tensor([step_idx] * latents.shape[0], device=latents.device)
        emb = self.step_embed(step_idx_tensor)

        # 3. Predict t
        inp = torch.cat([emb, stats], dim=1)
        t_norm = self.net(inp)

        # Scale to diffusion range (usually 0-1000)
        # We clamp slightly to avoid 0 or 1000 exactly for numerical stability
        return t_norm * 999.0 + 0.1

# ==========================================
# 2. Differentiable Sampling Logic
# ==========================================
class DifferentiableDiffusionHandler:
    """
    Helper to perform diffusion steps where 't' is a continuous variable
    that allows backpropagation.
    """
    def __init__(self, pipe):
        self.unet = pipe.unet
        self.scheduler = pipe.scheduler
        self.alphas_cumprod = self.scheduler.alphas_cumprod.to(pipe.device)

    def get_alpha_sigma(self, t):
        """
        Interpolates alpha_cumprod for a continuous timestep t.
        This allows gradients to flow from the loss -> latent -> alpha -> t.
        """
        # t is shape [Batch, 1]
        t = t.squeeze()

        # Indices for interpolation
        low_idx = t.floor().long().clamp(0, len(self.alphas_cumprod)-2)
        high_idx = low_idx + 1

        alpha_low = self.alphas_cumprod[low_idx]
        alpha_high = self.alphas_cumprod[high_idx]

        # Weights for interpolation
        w = t - low_idx.float()

        # Linear interpolation of alpha_cumprod
        # (Log-linear interpolation is often more accurate for diffusion,
        # but linear is sufficient for a demo)
        alpha_t = (1 - w) * alpha_low + w * alpha_high

        # Expand for broadcasting [Batch, 1, 1, 1]
        alpha_t = alpha_t.view(-1, 1, 1, 1)
        sigma_t = (1 - alpha_t) ** 0.5
        alpha_t = alpha_t ** 0.5 # We usually need sqrt(alpha_cumprod)

        return alpha_t, sigma_t

    def step(self, latents, t_now, t_next, text_embeddings):
        """
        One Differentiable DDIM Step.
        """
        # 1. Get noise levels for current t and next t
        alpha_now, sigma_now = self.get_alpha_sigma(t_now)
        alpha_next, sigma_next = self.get_alpha_sigma(t_next)

        # 2. Predict Noise (UNet is differentiable w.r.t input latents and t)
        # Note: diffusers unet accepts float t if provided
        noise_pred = self.unet(latents, t_now.squeeze(), encoder_hidden_states=text_embeddings).sample

        # 3. Predict x0 (clean image approximation)
        # latents = alpha_now * x0 + sigma_now * epsilon
        # x0 = (latents - sigma_now * epsilon) / alpha_now
        pred_x0 = (latents - sigma_now * noise_pred) / alpha_now

        # 4. Calculate direction to x_t_next
        # DDIM equation (deterministic)
        dir_xt = (1 - alpha_next**2)**0.5 * noise_pred # simplified sigma assumption

        # 5. Compute x_t_next
        prev_latents = alpha_next * pred_x0 + dir_xt

        return prev_latents

# ==========================================
# 3. Main Project Pipeline
# ==========================================

def run_project():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading Teacher (SD v1.5) on {device}...")
    # Load Standard SD 1.5
    model_id = "sd-legacy/stable-diffusion-v1-5"
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
    pipe = pipe.to("cuda")

    # Freeze Teacher
    pipe.unet.requires_grad_(False)
    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)

    # Enable Gradient Checkpointing to save memory on Colab
    pipe.unet.enable_gradient_checkpointing()

    # Define Student Settings
    K_STEPS = 4  # We want to compress generation into 4 steps
    student_scheduler = LearnedStepScheduler(num_steps=K_STEPS).to(device)

    # Optimizer for the scheduler
    optimizer = torch.optim.AdamW(student_scheduler.parameters(), lr=1e-3)

    # Differentiable Handler
    diff_handler = DifferentiableDiffusionHandler(pipe)

    # Dummy Training Data (In reality, you would load a dataset)
    prompts = [
        "A futuristic city with flying cars, cyberpunk style",
        "A photo of a cute corgi running in a field",
        "An oil painting of a cottage by a river",
        "A robot playing chess against a human"
    ]

    print("Starting Training Loop...")

    # Convert prompts to embeddings once (cache them)
    encoded_prompts = []
    for p in prompts:
        inputs = pipe.tokenizer(p, return_tensors="pt", padding="max_length", max_length=77, truncation=True).to(device)
        with torch.no_grad():
            emb = pipe.text_encoder(inputs.input_ids)[0]
        encoded_prompts.append(emb)

    # Training Loop
    epochs = 50 # Kept low for demo

    for epoch in range(epochs):
        epoch_loss = 0

        for text_emb in encoded_prompts:
            # Create a batch of 1 for simplicity
            text_emb = text_emb # [1, 77, 768]

            # --- 1. Teacher Ground Truth ---
            # Generate a "Perfect" target using the teacher (frozen)
            # For efficiency, we just target the Latents, not the pixels
            # In a real run, you might cache these targets beforehand.
            with torch.no_grad():
                # Generate random noise
                latents_shape = (1, 4, 64, 64)
                init_noise = randn_tensor(latents_shape, device=device, dtype=text_emb.dtype)

                # Run teacher (standard 50 steps) - Simplified call for demo
                teacher_latents = init_noise
                pipe.scheduler.set_timesteps(50)
                for t in pipe.scheduler.timesteps:
                    noise_pred = pipe.unet(teacher_latents, t, encoder_hidden_states=text_emb).sample
                    teacher_latents = pipe.scheduler.step(noise_pred, t, teacher_latents).prev_sample

            # --- 2. Student Process (K Steps) ---
            student_latents = init_noise.clone() # Start from same noise

            # We need to collect "trajectory" of timesteps
            # The student determines: t_0 -> t_1 -> t_2 -> t_3 -> 0

            for k in range(K_STEPS):
                # Predict current timestep t to use
                # Gradient flows through here!
                t_current = student_scheduler(k, student_latents)

                # Predict next timestep (Lookahead) or assume 0 for last step
                if k < K_STEPS - 1:
                    t_next = student_scheduler(k+1, student_latents)
                else:
                    t_next = torch.zeros_like(t_current) # Last step goes to 0

                # Enforce t_next < t_current (Time must flow backwards)
                # Soft constraint or simple ReLU logic
                # For this demo, we trust the optimizer or add a loss penalty,
                # but to prevent crash we can clamp:
                t_next = torch.min(t_next, t_current - 1).clamp(min=0)

                # Differentiable Sampling Step
                student_latents = diff_handler.step(student_latents, t_current, t_next, text_emb)

            # --- 3. Compute Loss ---
            # Compare final student latent to teacher final latent
            loss = F.mse_loss(student_latents, teacher_latents)

            # Optional: Add penalty if t_current are not descending

            # --- 4. Backprop ---
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")

    print("Training Complete!")
    return student_scheduler, pipe

# ==========================================
# 4. Inference / Demonstration
# ==========================================
def inference(student_model, pipe, prompt):
    device = pipe.device
    inputs = pipe.tokenizer(prompt, return_tensors="pt", padding="max_length", max_length=77, truncation=True).to(device)
    with torch.no_grad():
        text_emb = pipe.text_encoder(inputs.input_ids)[0]

    # Init Latents
    latents = randn_tensor((1, 4, 64, 64), device=device, dtype=text_emb.dtype)
    diff_handler = DifferentiableDiffusionHandler(pipe)

    print(f"Generating '{prompt}' in 4 steps...")

    with torch.no_grad():
        for k in range(4):
            t_curr = student_model(k, latents)
            if k < 3:
                t_next = student_model(k+1, latents)
                t_next = torch.min(t_next, t_curr - 1).clamp(min=0)
            else:
                t_next = torch.zeros_like(t_curr)

            print(f"  Step {k}: t_current={t_curr.item():.1f} -> t_next={t_next.item():.1f}")
            latents = diff_handler.step(latents, t_curr, t_next, text_emb)

    # Decode to Image
    latents = 1 / 0.18215 * latents
    image = pipe.vae.decode(latents).sample
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).float().numpy()

    return image[0]

# To Run:
# student, pipe = run_project()
# img = inference(student, pipe, "A photo of an astronaut riding a horse on mars")
# import matplotlib.pyplot as plt
# plt.imshow(img); plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline, DDIMScheduler
from diffusers.models.embeddings import TimestepEmbedding
import matplotlib.pyplot as plt
import numpy as np

# --- Configuration ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16  # Use float16 for memory efficiency
NUM_STUDENT_STEPS = 4  # K steps (The goal is to compress to this)
TEACHER_STEPS = 50     # Standard quality steps
LEARNING_RATE = 1e-4
BATCH_SIZE = 1         # Keep small for Colab VRAM

print(f"Running on {DEVICE} with K={NUM_STUDENT_STEPS}")

# --- 1. The Student Scheduler (The "Brain") ---
class LearnedScheduler(nn.Module):
    def __init__(self, latent_dim=4, hidden_dim=64):
        super().__init__()
        # Input: Mean of Latents (4) + Std of Latents (4) + Step Index (1)
        input_dim = latent_dim * 2 + 1

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1), # Outputs a value between 0 and 1
            nn.Sigmoid()
        )

    def forward(self, latents, step_index):
        # Extract simple statistics from latents to keep MLP small
        # Latents shape: (B, 4, 64, 64)
        mean = torch.mean(latents, dim=[2, 3]) # (B, 4)
        std = torch.std(latents, dim=[2, 3])   # (B, 4)

        # Normalize step index (0 to 1)
        step_tensor = torch.tensor([step_index / NUM_STUDENT_STEPS], device=latents.device).expand(latents.shape[0], 1)

        inp = torch.cat([mean, std, step_tensor], dim=1).float() # MLP usually needs float32

        # Predict normalized timestep (0.0 to 1.0)
        pred_norm = self.net(inp)

        # Scale to diffusion timesteps (0 to 1000)
        # We output high values (noise) first, then low values.
        # However, the network can learn any order. We initialize it to roughly linear decay logic.
        return pred_norm * 1000

# --- 2. Differentiable Solver Helper ---
# We need a custom way to convert float timesteps to embeddings so gradients flow
def get_timestep_embedding(timestep, dim=320, max_period=10000):
    # timestep: float tensor (B, 1)
    # This mimics the standard sinusoidal embedding but keeps it differentiable
    half = dim // 2
    freqs = torch.exp(
        -torch.log(torch.tensor(max_period)) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timestep.device)

    args = timestep.float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding

# --- 3. Main Project Class ---
class DistilledDiffusionProject:
    def __init__(self):
        # Load SD v1.5
        print("Loading Stable Diffusion...")
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=DTYPE
        ).to(DEVICE)

        # Freeze EVERYTHING in the teacher
        self.pipe.vae.requires_grad_(False)
        self.pipe.text_encoder.requires_grad_(False)
        self.pipe.unet.requires_grad_(False)

        # Enable gradient checkpointing to save VRAM (Critical for training loop)
        self.pipe.unet.enable_gradient_checkpointing()

        # The Student Component
        self.student_scheduler = LearnedScheduler().to(DEVICE)
        self.optimizer = torch.optim.AdamW(self.student_scheduler.parameters(), lr=LEARNING_RATE)

        # Standard Scheduler for the Teacher (Ground Truth)
        self.teacher_scheduler = DDIMScheduler.from_config(self.pipe.scheduler.config)
        self.teacher_scheduler.set_timesteps(TEACHER_STEPS)

    def get_teacher_target(self, prompt):
        """Generates the 'Ground Truth' image using 50 steps."""
        with torch.no_grad():
            output = self.pipe(prompt, num_inference_steps=TEACHER_STEPS, output_type="latent")
            return output.images # Returns latents (B, 4, 64, 64)

    def run_student_loop(self, prompt):
        """Runs the K-step loop where the Scheduler decides the timesteps."""
        # 1. Setup Text Embeddings
        text_inputs = self.pipe.tokenizer(prompt, padding="max_length", max_length=self.pipe.tokenizer.model_max_length, truncation=True, return_tensors="pt")
        text_embeddings = self.pipe.text_encoder(text_inputs.input_ids.to(DEVICE))[0]

        # 2. Initial Random Noise
        latents = torch.randn((1, 4, 64, 64), device=DEVICE, dtype=DTYPE)

        # 3. The Learned Loop
        # We perform K updates. The scheduler predicts 't' for the UNet.
        # We use a simplified update rule (DDIM-like) that allows gradients.

        for k in range(NUM_STUDENT_STEPS):
            # A. Student Scheduler predicts which timestep to query
            # We want gradients to flow through 't_pred'
            t_pred = self.student_scheduler(latents, k) # Shape (1, 1)

            # B. Get Differentiable Time Embedding
            # Standard UNet expects (B,) long, but we force (B, 1, 320) embedding
            t_emb = get_timestep_embedding(t_pred, dim=320).to(dtype=DTYPE)
            t_emb = self.pipe.unet.time_embedding(t_emb) # Project to correct internal dim (1280)

            # C. Predict Noise using Teacher UNet
            # Note: We pass the constructed embedding directly
            noise_pred = self.pipe.unet(
                latents,
                t_pred.view(-1), # Passed for internal logic, but embedding overrides mostly
                encoder_hidden_states=text_embeddings,
                timestep_cond=t_emb # Inject our differentiable time
            ).sample

            # D. The Update Step (Simplified DDIM step for differentiability)
            # x_prev = x_t - noise_pred * step_size (Rough Approximation for learning)
            # In a real rigorous paper, we would map t_pred to Alpha_cumprod(t)
            # Here we learn the trajectory directly.

            # We treat the output of scheduler as a "step magnitude" implicitly
            step_size = 1.0 / NUM_STUDENT_STEPS
            latents = latents - (noise_pred * step_size)

        return latents

    def train_step(self, prompt="A photo of a cute corgi"):
        self.optimizer.zero_grad()

        # 1. Get Teacher Target (Frozen, no grad)
        target_latents = self.get_teacher_target(prompt)

        # 2. Run Student (With Grad)
        student_latents = self.run_student_loop(prompt)

        # 3. Compute Loss (MSE in Latent Space)
        loss = F.mse_loss(student_latents, target_latents)

        # 4. Update
        loss.backward()
        self.optimizer.step()

        return loss.item(), student_latents, target_latents

    def decode_image(self, latents):
        """Helper to turn latents into PIL image"""
        with torch.no_grad():
            latents = 1 / 0.18215 * latents
            image = self.pipe.vae.decode(latents).sample
            image = (image / 2 + 0.5).clamp(0, 1)
            image = image.cpu().permute(0, 2, 3, 1).numpy()[0]
            return image

# --- 4. Execution Loop ---

project = DistilledDiffusionProject()

# Training for a few iterations to demonstrate
prompts = [
    "A futuristic city with flying cars",
    "A cozy cabin in the snowy woods",
    "Portrait of a warrior with armor",
    "A bowl of fresh fruit on a table"
] * 5 # 20 iterations

loss_history = []

print("Starting Training...")
for i, prompt in enumerate(prompts):
    loss, s_lat, t_lat = project.train_step(prompt)
    loss_history.append(loss)
    print(f"Iter {i+1}/{len(prompts)} | Loss: {loss:.4f}")

    # Visualize the last one
    if i == len(prompts) - 1:
        print("Decoding final images...")
        img_student = project.decode_image(s_lat)
        img_teacher = project.decode_image(t_lat)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.title("Student (4 Steps)")
        plt.imshow(img_student)
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.title("Teacher (50 Steps)")
        plt.imshow(img_teacher)
        plt.axis('off')
        plt.show()

# Plot Loss
plt.plot(loss_history)
plt.title("Distillation Loss")
plt.xlabel("Iteration")
plt.ylabel("MSE Loss")
plt.show()